# 🔤 From Text to Attention

By the end of this notebook, you'll understand:
1. How text becomes tokens (and why it matters)
2. How tokens become meaningful vectors (embeddings)
3. How position information gets added
4. How attention actually works — coded from scratch

Let's start!

# Part 1 : tokenization 

tiktoken is a tokenizer library created by OpenAI. Its main job is to convert text into tokens (and back), using the same tokenization logic as OpenAI language models.

In simple terms

tiktoken tells you:

How a piece of text is split into tokens

How many tokens a text will use

Which tokens map to which numbers (IDs)

This is crucial because LLMs don’t read text directly—they read tokens, and APIs have token limits and costs based on them.

What you use tiktoken for
1. Counting tokens accurately

Before sending text to an LLM, you can check:

Will this exceed the model’s context limit?

How expensive will this request be?

import tiktoken


enc = tiktoken.encoding_for_model("gpt-4")
tokens = enc.encode("Hello world!")
len(tokens)
2. Matching OpenAI model tokenization

Different models tokenize text differently. tiktoken ensures your token counts match exactly what the model will use.

Examples:

"hello" → 1 token

"hello " → might be 2 tokens

Emojis, whitespace, and non-English text behave differently

3. Debugging prompts

If a prompt behaves oddly, token inspection can reveal:

Hidden whitespace

Unexpected token splits

Why a prompt is longer than expected

4. Chunking long documents

When embedding or summarizing large texts, tiktoken helps you:

Split text into chunks that fit within model limits

Avoid cutting text mid-token

In [2]:
!pip install tiktoken -q

In [2]:
import sys
!{sys.executable} -m pip install tiktoken -q

'c:\Users\manas\OneDrive\Desktop\DL' is not recognized as an internal or external command,
operable program or batch file.


## 1.1 Basic Tokenization

Let's see how text gets converted to numbers.

In [4]:
import tiktoken 
import numpy as np 

# load tokenizer GPT-4 tokenizer    
tokenizer = tiktoken.get_encoding("cl100k_base")

print(f"vocab size:{tokenizer.n_vocab}.")


vocab size:100277.


In [11]:
tokenizer = tiktoken.encoding_for_model("gpt-4")
tokenizer.encode("hi there")
tokenizer.encode("Harshwardhan Tiwari") , tokenizer.encode("Snehal Soni")


# this library gets us the token logic used in the exact gpt-4 model 
# text into numbers 

([27588, 939, 1637, 10118, 23126, 86, 2850], [50, 818, 12130, 12103, 72])

In [13]:
len(tokenizer.encode("Harshwardhan Tiwari")) , len(tokenizer.encode("Snehal Soni"))

(7, 5)

In [17]:
# let's userstand in more detail like how the things work 
examples = ["Hellow world" , "don't" , "ai hai bhaijaan" , "I love you" , "Supercalifragilisticexpialidicious" , "cafe" , "   spaces    "]

print("let's see how the different tokens are encoded ")

for text in examples:
    token = tokenizer.encode(text)
    print("-" * 50)
    print(token)
    print(f"token shape {token} and length {len(token)}")
    print(" tokens are made as : {token}")

    pieces = [tokenizer.decode([t]) for t in token]
    print(f"pieces are made as  : {pieces}")
    print("-"*50)
    print("\n\n")

let's see how the different tokens are encoded 
--------------------------------------------------
[39, 5412, 1917]
token shape [39, 5412, 1917] and length 3
 tokens are made as : {token}
pieces are made as  : ['H', 'ellow', ' world']
--------------------------------------------------



--------------------------------------------------
[15357, 956]
token shape [15357, 956] and length 2
 tokens are made as : {token}
pieces are made as  : ['don', "'t"]
--------------------------------------------------



--------------------------------------------------
[2192, 47151, 293, 26279, 5697, 276]
token shape [2192, 47151, 293, 26279, 5697, 276] and length 6
 tokens are made as : {token}
pieces are made as  : ['ai', ' hai', ' b', 'hai', 'ja', 'an']
--------------------------------------------------



--------------------------------------------------
[40, 3021, 499]
token shape [40, 3021, 499] and length 3
 tokens are made as : {token}
pieces are made as  : ['I', ' love', ' you']
----------

## Why tokenization matters ? 
token counts affects :
api cost 
context limits the gpt -4 has context limit of 128k tokens 
model behavior (some tasks break across token)

In [18]:
# Compare token efficiency across different content types

test_cases = {
    "English prose": "The quick brown fox jumps over the lazy dog.",
    "Python code": "def hello():\n    print('Hello, world!')",
    "JSON": '{"name": "Alice", "age": 30, "city": "NYC"}',
    "Numbers": "1234567890 9876543210 1111111111",
    "URL": "https://www.example.com/path/to/page?query=value",
}

print("Token efficiency comparison:\n")
for name, text in test_cases.items():
    tokens = tokenizer.encode(text)
    chars = len(text)
    ratio = chars / len(tokens)
    print(f"{name}:")
    print(f"  {chars} chars → {len(tokens)} tokens ({ratio:.1f} chars/token)")
    print()

Token efficiency comparison:

English prose:
  44 chars → 10 tokens (4.4 chars/token)

Python code:
  39 chars → 11 tokens (3.5 chars/token)

JSON:
  43 chars → 19 tokens (2.3 chars/token)

Numbers:
  32 chars → 14 tokens (2.3 chars/token)

URL:
  48 chars → 11 tokens (4.4 chars/token)



In [19]:
# why does the llm struggle with the letter counting tasks 

from traceback import print_tb


word = "supercalifragilisticexpialidocious"
print(f"Word: {word}")
print(f"token {tokenizer.encode(word)}")
print(f"token lenght {len(tokenizer.encode(word))}")

print(f"pieces : {[tokenizer.decode([t]) for t in tokenizer.encode(word)]}")
print(f"couting the number of i in the word : {word.count('i')}")
print("model sees these pieces and not the individual charectors ")


Word: supercalifragilisticexpialidocious
token [13066, 3035, 278, 333, 4193, 321, 4633, 4683, 532, 307, 78287]
token lenght 11
pieces : ['sup', 'erc', 'al', 'if', 'rag', 'il', 'istic', 'exp', 'ial', 'id', 'ocious']
couting the number of i in the word : 7
model sees these pieces and not the individual charectors 


In [20]:
# Another example: reversing words

word = "hello"
tokens = tokenizer.encode(word)
print(f"'{word}' → tokens: {[tokenizer.decode([t]) for t in tokens]}")
print()
print("If 'hello' is ONE token, the model can't easily reverse it.")
print("It would need to decompose something it sees as atomic.")

'hello' → tokens: ['hello']

If 'hello' is ONE token, the model can't easily reverse it.
It would need to decompose something it sees as atomic.
